In [1]:
from clickhouse_driver import Client

client = Client(host='clickhouse', port=9000)

result = client.execute('SELECT version()')
print(f'ClickHouse version: {result[0][0]}')

ClickHouse version: 23.8.16.16


In [2]:
result = client.execute("""
    SELECT name, engine
    FROM system.tables
    WHERE database = 'default'
    ORDER BY name
""")

for name, engine in result:
    print(name, engine)

currency_rates ReplacingMergeTree
dict_account_level Dictionary
dict_account_level_source MergeTree
dict_country Dictionary
dict_country_source MergeTree
dict_merchant_category Dictionary
dict_merchant_category_source MergeTree
kafka_transactions_invalid Kafka
kafka_transactions_valid Kafka
mv_transactions_invalid MaterializedView
mv_transactions_valid MaterializedView
transactions_invalid ReplacingMergeTree
transactions_valid ReplacingMergeTree


In [18]:
import pandas as pd
pd.DataFrame(
    client.execute("select * from currency_rates FINAL"), 
    columns=('dt', 'currency', 'rate_to_rub', 'nominal', 'loaded_at')
).sort_values(by='dt').tail(10)

Error on clickhouse:9000 ping: Unexpected EOF while reading bytes
Connection was closed, reconnecting.
Error on socket shutdown: [Errno 107] Transport endpoint is not connected


,dt,currency,rate_to_rub,nominal,loaded_at
15005,2026-04-09,GBP,103.7297,1,2026-04-12 06:34:44
15010,2026-04-11,RUB,1,1,2026-04-12 00:23:32
15011,2026-04-11,USD,76.9724,1,2026-04-12 00:23:32
15008,2026-04-11,EUR,90.012,1,2026-04-12 00:23:32
15009,2026-04-11,GBP,103.2585,1,2026-04-12 00:23:32
15014,2026-04-12,RUB,1,1,2026-04-12 09:05:02
15015,2026-04-12,USD,76.9724,1,2026-04-12 09:05:02
15012,2026-04-12,EUR,90.012,1,2026-04-12 09:05:02
15013,2026-04-12,GBP,103.2585,1,2026-04-12 09:05:02
15016,2026-12-01,RUB,1,1,2026-04-12 00:24:24


In [4]:
merchant_categories = [
    ('other', 'Other', 9, 0),
    ('personal_transfer', 'Personal Transfer', 8, 1),
    ('ATM', 'ATM Withdrawal', 7, 0),
    ('utilities', 'Utilities & Bills', 4, 1),
    ('entertainment', 'Entertainment', 4, 1),
    ('transport', 'Transport', 2, 0),
    ('food', 'Food & Dining', 2, 0),
    ('retail', 'Retail Shopping', 2, 0),
]

client.execute("TRUNCATE TABLE dict_merchant_category_source")

client.execute(
    'INSERT INTO dict_merchant_category_source '
    '(category_code, category_name, risk_score, is_online) VALUES',
    merchant_categories
)

8

In [5]:
result = client.execute(
    'SELECT category_code, category_name, risk_score, is_online '
    'FROM dict_merchant_category_source '
    'ORDER BY risk_score DESC'
)

pd.DataFrame(result)

,0,1,2,3
0,other,Other,9,0
1,personal_transfer,Personal Transfer,8,1
2,ATM,ATM Withdrawal,7,0
3,entertainment,Entertainment,4,1
4,utilities,Utilities & Bills,4,1
5,food,Food & Dining,2,0
6,retail,Retail Shopping,2,0
7,transport,Transport,2,0


In [6]:
countries = [
    ('US', 'United States',  'Americas', 1),
    ('GB', 'United Kingdom', 'Europe',   1),
    ('FR', 'France',         'Europe',   1),
    ('DE', 'Germany',        'Europe',   1),
    ('JP', 'Japan',          'Asia',     2),
    ('RU', 'Russia',         'Europe',   3),
]

client.execute('TRUNCATE TABLE dict_country_source')
client.execute(
    'INSERT INTO dict_country_source '
    '(country_code, country_name, region, risk_level) VALUES',
    countries
)

6

In [7]:
result = client.execute(
    'SELECT country_code, country_name, region, risk_level '
    'FROM dict_country_source '
    'ORDER BY risk_level, country_code'
)

pd.DataFrame(result)

,0,1,2,3
0,DE,Germany,Europe,1
1,FR,France,Europe,1
2,GB,United Kingdom,Europe,1
3,US,United States,Americas,1
4,JP,Japan,Asia,2
5,RU,Russia,Europe,3


In [8]:
from decimal import Decimal

account_levels = [
    ('standard', 'Standard', Decimal('10000.00'),  Decimal('50000.00')),
    ('premium',  'Premium',  Decimal('25000.00'),  Decimal('150000.00')),
    ('vip',      'VIP',      Decimal('100000.00'), Decimal('500000.00')),
]

client.execute('TRUNCATE TABLE dict_account_level_source')
client.execute(
    'INSERT INTO dict_account_level_source '
    '(level_code, level_name, daily_limit_rub, monthly_limit_rub) VALUES',
    account_levels
)

3

In [9]:
result = client.execute(
    'SELECT level_code, level_name, daily_limit_rub, monthly_limit_rub '
    'FROM dict_account_level_source '
    'ORDER BY daily_limit_rub'
)

pd.DataFrame(result)

,0,1,2,3
0,standard,Standard,10000,50000
1,premium,Premium,25000,150000
2,vip,VIP,100000,500000


In [10]:
client.execute("SYSTEM RELOAD DICTIONARY dict_account_level")
client.execute("SYSTEM RELOAD DICTIONARY dict_merchant_category")
client.execute("SYSTEM RELOAD DICTIONARY dict_country")

[]